## Libraries

In [15]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

## Finding the Number of Trials

In [16]:
n_trials = 0
for path in os.listdir("Extracted_Data/MM_S1/neural_data_PMd"):
    if os.path.isfile(os.path.join("Extracted_Data/MM_S1/neural_data_PMd", path)):
        n_trials += 1

print(n_trials)

496


## Finding how many neurons are in this monkey

In [17]:
neuro_df1 = pd.read_csv("Extracted_Data/MM_S1/neural_data_PMd/neural_data_PMd_reach1.csv")

n_neurons = len(neuro_df1.columns) - 1
print(n_neurons)

94


## Adjust Time delay

In [ ]:
timeshift_delay = 6
timeDelayed = True

In [ ]:
# Get the "base" dataframe to concat from
neuro_df1 = pd.read_csv("Extracted_Data/MM_S1/neural_data_PMd/neural_data_PMd_reach1.csv")
kinem_df1 = pd.read_csv("Extracted_Data/MM_S1/kinematic_data/reach1.csv")
merged_df = pd.concat((neuro_df1, kinem_df1), axis=1)

# Merge all data
for i in range(1, n_trials):

    #Get neural data
    neuro_df = pd.read_csv(f"Extracted_Data/MM_S1/neural_data_PMd/neural_data_PMd_reach{i+1}.csv")
    kinem_df = pd.read_csv(f"Extracted_Data/MM_S1/kinematic_data/reach{i+1}.csv")

    # Timeshift (Move 1 time bin behind) 
    if timeDelayed:
        neuro_df = neuro_df.iloc[timeshift_delay:]

    # Merge neuro and kin data
    merge_ = pd.concat((neuro_df, kinem_df), axis=1)
    merge_ = merge_.dropna()

    # Merge to main
    merged_df = pd.concat((merged_df, merge_), axis = 0)

## Linear Regression

In [30]:
y = merged_df["x_acceleration"]
x = merged_df.loc[:, 'Neuron1':'Neuron94']
X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size = 0.2) 
model = LinearRegression()
results = model.fit(X_train, Y_train)
print(model.score(X_test, Y_test)) 

0.024338876159981537


# Brute Force Code to Determine What Time Shift Produces Maximum Correlation

In [31]:
max = 0
max_index = 0

neuro_df1 = pd.read_csv("Extracted_Data/MM_S1/neural_data_PMd/neural_data_PMd_reach1.csv")
kinem_df1 = pd.read_csv("Extracted_Data/MM_S1/kinematic_data/reach1.csv")

for j in range(20):
    merged_df = pd.concat((neuro_df1, kinem_df1), axis=1)

    for i in range(1, n_trials):
        neuro_df = pd.read_csv(f"Extracted_Data/MM_S1/neural_data_PMd/neural_data_PMd_reach{i+1}.csv")
        kinem_df = pd.read_csv(f"Extracted_Data/MM_S1/kinematic_data/reach{i+1}.csv")

        # Timeshift (Move 1 time bin behind) 
        neuro_df = neuro_df.iloc[j:]

        # Merge neuro and kin data
        merge_ = pd.concat((neuro_df, kinem_df), axis=1)
        merge_ = merge_.dropna()

        # Merge to main
        merged_df = pd.concat((merged_df, merge_), axis = 0)

    y = merged_df["x_acceleration"]
    x = merged_df.loc[:, 'Neuron1':'Neuron94']
    X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size = 0.2) 
    model = LinearRegression()
    model.fit(X_train, Y_train)
    
    if model.score(X_test, Y_test) > max:
        max = model.score(X_test, Y_test)
        max_index = j

print(f"{max} at {max_index}")

0.025977380356033253 at 6
